In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT = Path('BioReasoningChallenge/data')
LOCAL_DATA = PROJECT_ROOT / 'data'

DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_DATA
print(f'Using data directory: {DATA_DIR}')

if Path('/kaggle/input').exists():
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(os.path.join(dirname, filename))


In [ ]:
SEEDS = [42, 43, 44]

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

prompt_candidates = [
    Path('/kaggle/input/datasets/nident/prompts/prompt.txt'),
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '04_combined_de_first_fewshot_strict.txt',
    PROJECT_ROOT / 'prompt_strategies' / 'track_a' / '00_zero_shot_strict.txt',
]
default_prompt_path = next((path for path in prompt_candidates if path.exists()), None)
if default_prompt_path is None:
    raise FileNotFoundError('No prompt template found in Kaggle input or local prompt_strategies/track_a')

default_prompt = default_prompt_path.read_text(encoding='utf-8')

print(f'Using prompt template: {default_prompt_path}')
print(default_prompt[:1000])
print(train.columns.tolist())
print(train.head(2))
print(test.head(2))


In [ ]:
import os
from pprint import pprint
from openai import OpenAI

api_key = os.getenv('OPENROUTER_API_KEY')

if not api_key:
    print('OPENROUTER_API_KEY is not set; skipping live OpenRouter smoke test.')
else:
    client = OpenAI(
        base_url='https://openrouter.ai/api/v1',
        api_key=api_key,
    )

    sample = train.iloc[0]
    prompt = default_prompt.format(
        pert=sample['pert'],
        gene=sample['gene'],
        cell_desc='Mouse bone marrow-derived macrophages (BMDMs) are primary immune cells differentiated from bone marrow precursors using M-CSF.',
    )

    response = client.chat.completions.create(
        model='openai/gpt-oss-120b:free',
        messages=[{'role': 'user', 'content': prompt}],
        extra_body={'reasoning': {'enabled': True}},
    )

    pprint(response.choices[0].message.content)
